In [1]:

import heapq
import networkx as nx
import matplotlib.pyplot as plt
from math import sqrt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

#Graf i pozycje (przydadzą się do heurystyki)
weights = {
    ('A','B'): 2,
    ('A','C'): 4,
    ('B','C'): 1,
    ('B','D'): 7,
    ('C','D'): 3,
    ('C','E'): 1,
    ('D','E'): 2,
    ('D','F'): 5,
    ('E','F'): 7
}

positions = {
    'A': (0,0),
    'B': (1,1),
    'C': (1,0),
    'D': (2,1),
    'E': (2,0),
    'F': (3,0.5)
}

G = nx.Graph()
for (u,v), w in weights.items():
    G.add_edge(u,v,weight=w)

min_weight = min(weights.values())


#Heurystyka
def heuristic(a,b):
    x1,y1 = positions[a]
    x2,y2 = positions[b]
    return sqrt((x1-x2)**2 + (y1-y2)**2) * min_weight #w celu dostosowania skali odległości euklidesowej do skali kosztów grafu
    #po przemnożeniu przez min_weight heurystyka lepiej odpowiada rzeczywistym kosztom przejścia między węzłami, każdy krok na mapie kosztuje co najmniej tyle ile najtańsza krawędź


#Funkcja generująca kroki A*
def a_star_steps(graph, start, goal):
    open_set = []    #lista punktów do sprawdzenia
    heapq.heappush(open_set, (0, start))   #wrzuca element do kolejki priorytetowej
    came_from = {}   #zapamięta, skąd przyszliśmy
    g_score = {node: float('inf') for node in graph.nodes} #koszt dojścia do każdego wierzchołka - nie znamy jeszcze drogi - ustawiamy nieskończoność
    g_score[start] = 0
    closed_set = set() #zbiór odwiedzonych punktów
    steps = [] #kroki do animacji

    while open_set:  #gdy lista punktów do sprawdzenia niepusta
        f_current, current = heapq.heappop(open_set) #heappop wyciąga element o najmniejszym parametrze
        if current in closed_set:
            continue
        closed_set.add(current) #dodajemy do odwiedzonych

        #uwzględniamy w animacji
        steps.append((current, closed_set.copy(), None))

        if current == goal:
            #Rekonstrukcja ścieżki (od końca zaczynając, cofamy się po rodzicach)
            path = [current]
            while current in came_from:
                current = came_from[current]
                path.append(current)
            path.reverse() #odwracamy by objąć dobry kierunek

            #animacja rysowania ścieżki krok po kroku
            for i in range(1, len(path)+1):
                partial_path = path[:i]
                steps.append((current, closed_set.copy(), partial_path))

            cost = g_score[goal] #koszt najkrótszej ścieżki

            return steps, path, cost

        for neighbor in graph.neighbors(current):  #sprawdzamy sąsiadów
            tentative_g = g_score[current] + graph[current][neighbor]['weight']  #koszt dojścia do sąsiada przez current
            if tentative_g < g_score[neighbor]:
                g_score[neighbor] = tentative_g #jeśli znaleźliśmy lepszą drogę, to aktualizujemy najlepszy koszt
                f = tentative_g + heuristic(neighbor, goal) #najlepsza ścieżka jako suma obu funkcji
                heapq.heappush(open_set, (f, neighbor)) #dodajemy do kolejki priorytetowej
                came_from[neighbor] = current #trzeba zapisać, skąd przyszliśmy

    return steps


#Funkcja rysująca krok animacji

def update(frame):
    ax.clear()
    current, closed, path = frame
    node_colors = []
    for node in G.nodes():
        if path and node in path:
            node_colors.append("#7B2CBF") #ścieżka
        elif node == start:
            node_colors.append("#2D6A4F") #start
        elif node == goal:
            node_colors.append("#560BAD") #cel
        elif node == current:
            node_colors.append("#52B788") #aktualny
        elif node in closed:
            node_colors.append("#4EA8DE") #odwiedzone
        else:
            node_colors.append("#CED4DA") #reszta

    nx.draw(G, positions, with_labels=True, node_color=node_colors, node_size=1200,
            font_size=12, edgecolors="black", linewidths=1.5, ax=ax)
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, positions, edge_labels=edge_labels, ax=ax)
    if path:
        path_edges = list(zip(path[:-1], path[1:]))
        nx.draw_networkx_edges(G, positions, edgelist=path_edges, edge_color="#7B2CBF", width=5, ax=ax)
    ax.set_title("Algorytm A*", fontsize=16)
    ax.axis("off")


#Przygotowanie animacji

start, goal = 'A','F'
steps, path, cost = a_star_steps(G, start, goal)

fig, ax = plt.subplots(figsize=(9,6))
ani = FuncAnimation(fig, update, frames=steps, interval=1000, repeat=False)
plt.close(fig)
# W Colabie używamy to_jshtml() aby animacja pokazała się w notebooku
display(HTML(ani.to_jshtml()))
print("\nWYNIK ALGORYTMU")
print("Najkrótsza ścieżka:", " → ".join(path))
print("Koszt ścieżki:", cost)



WYNIK ALGORYTMU
Najkrótsza ścieżka: A → B → C → E → F
Koszt ścieżki: 11


In [2]:
#inny przykład
weights = {
    ('A','B'): 3,
    ('A','C'): 2,
    ('B','D'): 4,
    ('B','E'): 6,
    ('C','D'): 1,
    ('C','F'): 7,
    ('D','E'): 2,
    ('D','F'): 3,
    ('E','G'): 4,
    ('F','G'): 2,
    ('G','H'): 3
}

positions = {
    'A': (0,2),
    'B': (1,3),
    'C': (1,1),
    'D': (2,2),
    'E': (3,3),
    'F': (3,1),
    'G': (4,2),
    'H': (5,2)
}

G = nx.Graph()

for (u, v), w in weights.items():
    G.add_edge(u, v, weight=w)

min_weight = min(weights.values())

start = 'A'
goal = 'H'

steps, path, cost = a_star_steps(G, start, goal)

fig, ax = plt.subplots(figsize=(10,6))
ani = FuncAnimation(fig, update, frames=steps, interval=1000, repeat=False)
plt.close(fig)
display(HTML(ani.to_jshtml()))
plt.close('all')

print("\nWYNIK ALGORYTMU")
print("Najkrótsza ścieżka:", " → ".join(path))
print("Koszt ścieżki:", cost)


WYNIK ALGORYTMU
Najkrótsza ścieżka: A → C → D → F → G → H
Koszt ścieżki: 11


In [3]:
#Łódź

# Przybliżone współrzędne geograficzne
geo_positions = {
    'Manufaktura': (51.7763, 19.4474),
    'Piotrkowska Centrum': (51.7592, 19.4585),
    'Dworzec Łódź Fabryczna': (51.7687, 19.4758),
    'EC1': (51.7670, 19.4788),
    'Politechnika Łódzka': (51.7473, 19.4555),
    'Atlas Arena': (51.7576, 19.4267),
    'Orientarium': (51.7596, 19.4018),
    'Lotnisko Lublinek': (51.7219, 19.3986)
}

# Zamiana współrzędnych geograficznych na układ wykresu
latitudes = [lat for lat, lon in geo_positions.values()]
longitudes = [lon for lat, lon in geo_positions.values()]

lat_min, lat_max = min(latitudes), max(latitudes)
lon_min, lon_max = min(longitudes), max(longitudes)

positions = {}

for place, (lat, lon) in geo_positions.items():
    x = (lon - lon_min) / (lon_max - lon_min) * 12
    y = (lat - lat_min) / (lat_max - lat_min) * 10
    positions[place] = (x, y)

#Tworzymy nowy graf
G = nx.Graph()

edges = [
    ('Manufaktura','Piotrkowska Centrum'),
    ('Manufaktura','Dworzec Łódź Fabryczna'),

    ('Piotrkowska Centrum','Politechnika Łódzka'),
    ('Piotrkowska Centrum','EC1'),

    ('Dworzec Łódź Fabryczna','EC1'),

    ('EC1','Politechnika Łódzka'),
    ('EC1','Atlas Arena'),

    ('Politechnika Łódzka','Atlas Arena'),

    ('Atlas Arena','Orientarium'),

    ('Orientarium','Lotnisko Lublinek'),

    ('Atlas Arena','Lotnisko Lublinek')
]

#Wagi liczone automatycznie z położenia punktów
for u, v in edges:
    x1, y1 = positions[u]
    x2, y2 = positions[v]

    distance = round(
        sqrt((x1 - x2)**2 + (y1 - y2)**2),
        1
    )

    G.add_edge(u, v, weight=distance)

#Aktualizacja zmiennej używanej przez heurystykę
weights = nx.get_edge_attributes(G, "weight")
min_weight = min(weights.values())


start = "Lotnisko Lublinek"
goal = "Dworzec Łódź Fabryczna"

steps, path, cost = a_star_steps(G, start, goal)

fig, ax = plt.subplots(figsize=(12,7))

ani = FuncAnimation(
    fig,
    update,
    frames=steps,
    interval=1200,
    repeat=False
)

plt.close(fig)
display(HTML(ani.to_jshtml()))
plt.close('all')

print("\nWYNIK ALGORYTMU")
print("Najkrótsza ścieżka:", " → ".join(path))
print("Koszt ścieżki:", round(cost, 2))


WYNIK ALGORYTMU
Najkrótsza ścieżka: Lotnisko Lublinek → Atlas Arena → EC1 → Dworzec Łódź Fabryczna
Koszt ścieżki: 16.3


In [4]:
#Polska

#Współrzędne geograficzne
geo_positions = {
    'Warszawa': (52.2297, 21.0122),
    'Kraków': (50.0647, 19.9450),
    'Gdańsk': (54.3520, 18.6466),
    'Wrocław': (51.1079, 17.0385),
    'Poznań': (52.4064, 16.9252),
    'Lublin': (51.2465, 22.5684),
    'Zakopane': (49.2992, 19.9496),
    'Łódź': (51.7592, 19.4560)
}

#Przeskalowanie współrzędnych
latitudes = [lat for lat, lon in geo_positions.values()]
longitudes = [lon for lat, lon in geo_positions.values()]

lat_min, lat_max = min(latitudes), max(latitudes)
lon_min, lon_max = min(longitudes), max(longitudes)

positions = {}
for place, (lat, lon) in geo_positions.items():
    x = (lon - lon_min) / (lon_max - lon_min) * 12
    y = (lat - lat_min) / (lat_max - lat_min) * 10
    positions[place] = (x, y)

#Graf i krawędzie

edges = [
    ('Warszawa','Łódź'),
    ('Łódź','Wrocław'),
    ('Łódź','Poznań'),
    ('Warszawa','Lublin'),
    ('Łódź','Kraków'),
    ('Kraków','Zakopane'),
    ('Kraków','Wrocław'),
    ('Poznań','Gdańsk'),
    ('Warszawa','Gdańsk')
]

G = nx.Graph()
for u, v in edges:
    x1, y1 = positions[u]
    x2, y2 = positions[v]
    distance = round(sqrt((x1-x2)**2 + (y1-y2)**2), 1)
    G.add_edge(u, v, weight=distance)

#Wartość minimalnej wagi do heurystyki
weights = nx.get_edge_attributes(G, "weight")
min_weight = min(weights.values())


start = "Zakopane"
goal = "Gdańsk"

steps, path, cost = a_star_steps(G, start, goal)

fig, ax = plt.subplots(figsize=(12,7))
ani = FuncAnimation(fig, update, frames=steps, interval=1200, repeat=False)

display(HTML(ani.to_jshtml()))
plt.close('all')
plt.close(fig)
print("\nWYNIK ALGORYTMU")
print("Najkrótsza ścieżka:", " → ".join(path))
print("Koszt ścieżki:", round(cost, 2))


WYNIK ALGORYTMU
Najkrótsza ścieżka: Zakopane → Kraków → Łódź → Warszawa → Gdańsk
Koszt ścieżki: 15.0


In [6]:
#graf skierowany i przeszkoda

geo_positions = {
    'Warszawa': (52.2297, 21.0122),
    'Kraków': (50.0647, 19.9450),
    'Gdańsk': (54.3520, 18.6466),
    'Wrocław': (51.1079, 17.0385),
    'Poznań': (52.4064, 16.9252),
    'Lublin': (51.2465, 22.5684),
    'Zakopane': (49.2992, 19.9496),
    'Łódź': (51.7592, 19.4560)
}

#Normalizacja współrzędnych do płaszczyzny
latitudes = [lat for lat, lon in geo_positions.values()]
longitudes = [lon for lat, lon in geo_positions.values()]
positions = {}
for place, (lat, lon) in geo_positions.items():
    x = (lon - min(longitudes)) / (max(longitudes) - min(longitudes)) * 12
    y = (lat - min(latitudes)) / (max(latitudes) - min(latitudes)) * 10
    positions[place] = (x, y)

#Graf skierowany
edges = [
    ('Zakopane','Kraków'),
    ('Łódź','Kraków'),
    ('Łódź','Warszawa'),
    ('Warszawa','Gdańsk'),
    ('Kraków','Wrocław'),
    ('Kraków','Warszawa'),
    ('Wrocław','Poznań'),
    ('Poznań','Gdańsk')
]

G = nx.DiGraph()
for u,v in edges:
    x1,y1 = positions[u]
    x2,y2 = positions[v]
    distance = round(sqrt((x1-x2)**2 + (y1-y2)**2),1)
    G.add_edge(u,v,weight=distance)

#blokujemy krawędź Łódź -> Warszawa
blocked_edges_nodes = set()
if G.has_edge('Łódź','Warszawa'):
    G.remove_edge('Łódź','Warszawa')
    blocked_edges_nodes.add('Łódź')

#Start i cel
start = 'Zakopane'
goal = 'Gdańsk'

#wywołanie
steps, path, cost = a_star_steps(G, start, goal)

#animacja
fig, ax = plt.subplots(figsize=(12,7))
ani = FuncAnimation(fig, update, frames=steps, interval=1200, repeat=False)
from IPython.display import HTML
display(HTML(ani.to_jshtml()))
plt.close('all')
plt.close(fig)

print("\nWYNIK ALGORYTMU")
if path:
    print("Najkrótsza ścieżka:", " → ".join(path))
    print("Koszt ścieżki:", round(cost,2))
else:
    print("Nie znaleziono ścieżki (przeszkoda blokuje dostęp).")


WYNIK ALGORYTMU
Najkrótsza ścieżka: Zakopane → Kraków → Warszawa → Gdańsk
Koszt ścieżki: 12.9
